In [2]:
import os
import numpy as np
import torch
import cv2
from torch import nn
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

In [ ]:
! nvidia-smi

In [2]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.168  Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
Setup complete  (16 CPUs, 31.9 GB RAM, 223.8/476.4 GB disk)


In [2]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

configure

In [3]:
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'  #r'yolo11m.pt'
PATH_VIDEO_TEST = r'E:\TRACKIING\video\IMG_2529_.mp4' #DJIG0103.mp4'

## Export модели 


yolo11n-cls для reid, нужны доп библиотеки

In [ ]:
# ! pip uninstall tensorrt nvidia-tensorrt
# ! pip cache purge
# ! pip install tensorrt==10.1.*

^C


In [4]:
import tensorrt
tensorrt.__version__

'10.13.3.9'

In [5]:
model_cls = YOLO('yolo11n-cls.pt')

In [6]:
head = model_cls.model.model[-1]
pool = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(start_dim=1))
pool.f, pool.i = head.f, head.i
model_cls.model.model[-1] = pool

model_cls.export(format='engine', half=True, dynamic=True, batch=32)

WARNING TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.3.168  Python-3.10.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
YOLO11n-cls summary (fused): 45 layers, 1,197,064 parameters, 0 gradients, 2.9 GFLOPs

PyTorch: starting from 'yolo11n-cls.pt' with input shape (32, 3, 224, 224) BCHW and output shape(s) (32, 256) (5.5 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success  2.2s, saved as 'yolo11n-cls.onnx' (4.6 MB)

TensorRT: starting export with TensorRT 10.13.3.9...
TensorRT: input "images" with shape(-1, 3, -1, -1) DataType.FLOAT
TensorRT: output "output0" with shape(-1, 256) DataType.FLOAT
TensorRT: building FP16 engine as yolo11n-cls.engine
TensorRT: export success  363.6s, saved as 'yolo11n-cls.engine' (4.8 MB)

Export complete (364.8s)
Results saved to C:\TASK_DETECT_DRONE\CODES_DETECT_DRONE\YOLO+calc_distance
Predict:         yolo predict task=classify model=yolo11n-cl

'yolo11n-cls.engine'

### export detectora in onnx

In [8]:
model = YOLO(PATH_WEIGHT)
model.export(format='onnx',
             imgsz=[720,1024],
             half = True,
             dynamic = True,
             batch=1,
             device=device,
             nms = True)

WARNING imgsz=[720, 1024] must be multiple of max stride 32, updating to [736, 1024]
WARNING 'dynamic=True' model with 'nms=True' requires max batch size, i.e. 'batch=16'
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'weights_sdu_v3\best_sdu_v3.pt' with input shape (1, 3, 736, 1024) BCHW and output shape(s) (1, 300, 6) (49.6 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success  10.6s, saved as 'weights_sdu_v3\best_sdu_v3.onnx' (49.5 MB)

Export complete (11.9s)
Results saved to C:\TASK_DETECT_DRONE\CODES_DETECT_DRONE\YOLO+calc_distance\weights_sdu_v3
Predict:         yolo predict task=detect model=weights_sdu_v3\best_sdu_v3.onnx imgsz=736,1024 half 
Validate:        yolo val task=detect model=weights_sdu_v3\best_sdu_v3.onnx imgsz=736,1024 data=data_yaml_sdu_v3.yaml half WARNING  non-PyTorch val requires square images, 'imgsz=[736, 1024]' will not work. Use export '

'weights_sdu_v3\\best_sdu_v3.onnx'

define function

In [9]:
def load_model(path_weight):
    if os.path.isfile(path_weight):
        model = YOLO(path_weight)
        return model
    else:
        print('Weight model not load')

In [10]:
current_tracks = []
selected_id = None

def mouse_click(event, x, y, flags, param):
    global selected_id, frame

    if event == cv2.EVENT_LBUTTONDOWN:
        # print(f'Click to coordinate: ({x}, {y})')

        # Проверяем попал ли клик в какойто bb
        for bbox in current_tracks:
            
            x1, y1, x2, y2, track_id = map(int, bbox[:5])
            if x1 <= x < x2 and y1 <= y <= y2:
                print(f'Choice object: {bbox}')
                selected_id = int(track_id)
                print(f'Choice object with ID: {selected_id}')
                break

Process

with press key in object

In [14]:
current_tracks = []
selected_id = None

def mouse_click(event, x, y, flags, param):
    global selected_id, frame

    if event == cv2.EVENT_LBUTTONDOWN:
        # print(f'Click to coordinate: ({x}, {y})')

        # Проверяем попал ли клик в какойто bb
        for bbox in current_tracks:
            
            x1, y1, x2, y2, track_id = map(int, bbox[:5])
            if x1 <= x < x2 and y1 <= y <= y2:
                print(f'Choice object: {bbox}')
                selected_id = int(track_id)
                print(f'Choice object with ID: {selected_id}')
                break

cap = cv2.VideoCapture(0)  #PATH_VIDEO_TEST

model = load_model(r'weights_sdu_v3\best_sdu_v3.onnx')
cv2.namedWindow('YOLO11 Tracking')
cv2.setMouseCallback("YOLO11 Tracking", mouse_click)


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                          conf = 0.4,
                          iou = 0.2,
                          imgsz = 1024,
                          persist=True, 
                          verbose = False,
                         
                          tracker=r'conf_trackers\botsort_custom.yaml')[0]
    if results.boxes and results.boxes.is_track:
        boxesxy = results.boxes.xyxy.cpu()
        # boxes = results.boxes.xywh.cpu()
        track_ids = results.boxes.id.int().cpu().tolist()
        confideces = results.boxes.conf.cpu().tolist()
        clas = results.boxes.cls.cpu().tolist()
        
        #plot the tracks
        for box, track_id, conf, cl_ in zip(boxesxy, track_ids, confideces, clas):
            x, y, x2, y2 = map(int, box)
            current_tracks.append((x, y, x2, y2, track_id))
            color = (0, 0, 255)
            if selected_id is not None and selected_id == track_id:
                color_ = (255, 0, 0)
                cv2.rectangle(frame, (x, y), (x2, y2), color_, 2)
                cv2.putText(frame, f'Selected_{track_id}', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color_, 2)
            else:
                cv2.rectangle(frame, (x, y), (x2, y2), color, 1)
                cv2.putText(frame, f'{track_id}', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('r'):
                selected_id = None
                print('Drop select object...')
        
    
    cv2.imshow('YOLO11 Tracking', frame)

   

    if cv2.waitKey(2) & 0xFF == ord('q'):
        print('Press q')
        break
        

cap.release()
cv2.destroyAllWindows()


WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading weights_sdu_v3\best_sdu_v3.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider
WARNING 'source' is missing. Using 'source=C:\Users\Pc\anaconda3\envs\env_detect_drone\Lib\site-packages\ultralytics\assets'.
Choice object: (326, 183, 335, 203, 1)
Choice object with ID: 1
Drop select object...
Choice object: (392, 168, 399, 188, 5)
Choice object with ID: 5
Drop select object...
Choice object: (453, 188, 462, 212, 3)
Choice object with ID: 3
Press q


In [ ]:
track_history = defaultdict(lambda: [])
model = load_model(PATH_WEIGHT)

assert os.path.isfile(PATH_VIDEO_TEST), 'Not file vide'

cap = cv2.VideoCapture(PATH_VIDEO_TEST)  #PATH_VIDEO_TEST

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                          conf = 0.4,
                          iou = 0.2,
                          imgsz = 1024,
                          persist=True, 
                          verbose = False,
                         
                          tracker='botsort.yaml')[0]
    if results.boxes and results.boxes.is_track:
        boxes = results.boxes.xywh.cpu()
        track_ids = results.boxes.id.int().cpu().tolist()
 

        #visual the result on the frame
        frame = results.plot()

        #plot the tracks
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y))) # y, x center point
            if len(track) > 30: # retain 30 tracks for 30 frames
                track.pop(0)

            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=1)

        cv2.imshow('YOLO11 Tracking', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print('Press q')
            break
    else:
        print('Not boxes or is_track')
        continue

cap.release()
cv2.destroyAllWindows()


Press q


In [7]:
model = load_model(PATH_WEIGHT)

# assert os.path.isfile(PATH_VIDEO_TEST), 'Not file vide'

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break
    results = model.track(frame, 
                    conf = 0.4,
                    iou = 0.2,
                    imgsz = 1024,
                    verbose = False,
                    device=device,
                    tracker=r'conf_trackers\botsort_custom.yaml'
                    )[0]
    annotaited_frame = results.plot()
    
    cv2.imshow('YOLO11 Tracking', annotaited_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('Press q')
        break
    

cap.release()
cv2.destroyAllWindows()


Press q


Test from boxmot tracker Botsort

In [5]:
from boxmot import BotSort

In [6]:
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'
model = load_model(PATH_WEIGHT) # load YOLO model SDU my

In [24]:
PATH_WEIGHT_BOTSORT = r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'
tracker = BotSort(reid_weights=Path(PATH_WEIGHT_BOTSORT), device='cuda:0', half=False)

2025-09-23 10:01:58.669 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-23 10:01:58.671 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-23 10:01:58.930 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


In [22]:
def detect(frame):
    detection = []
    threshold_conf = 0.25
    result = model(frame,
                   imgsz=1024,
                   conf=threshold_conf,
                   iou=0.4,
                   device='cpu',
                   verbose=False)
    boxes = result[0].boxes.xyxy.cpu()
    confs = result[0].boxes.conf.cpu()
    idx_cls = [int(i) for i in list(result[0].boxes.cls.cpu())]
    for box, conf, id_cl in zip(boxes, confs, idx_cls):
        x1 = box[0]
        y1 = box[1]
        x2 = box[2]
        y2 = box[3]
        detection.append((x1, y1, x2, y2, conf, id_cl))
    return np.array(detection, dtype=np.float32)


process

In [ ]:
PATH_VIDEO_TEST =  0 #r'E:\TRACKIING\video\IMG_2529_.mp4'  #r'E:\TRACKIING\video\DJIG0103.mp4'

track_history = defaultdict(lambda: [])
color = {
    '0': (255, 0, 0),
    '1': (0, 255, 0),
    '2': (230, 230, 230),
    '3': (255, 0, 0),
}
cap = cv2.VideoCapture(PATH_VIDEO_TEST)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Not read frame!')
        break

    det = detect(frame)

    tracks = tracker.update(det, frame)

    for track in tracks:
        x1, y1, x2, y2, track_id, conf, id_c, _ = map(int, track[:8])
        color_bb = color.get(str(id_c), (0, 0, 0))
        cv2.rectangle(frame, (x1, y1), (x2, y2), color_bb, 1)
        
        cv2.putText(frame, f'{track_id}', (x1, y1 -10), cv2.FONT_HERSHEY_COMPLEX, 0.75, color_bb, 2)

        #save history move
        center = ((x1 + x2)//2, (y1 + y2)//2)
        track_history[track_id].append(center)

        # ploting trajectory
        # for i in range(1, len(track_history[track_id])):
        #     cv2.line(frame, track_history[track_id][i - 1], track_history[track_id][i], (0, 0, 255), 2)

    
    
    cv2.imshow('YOLO11 Tracking', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('Press q')
        break
    

cap.release()
cv2.destroyAllWindows()


# SELF Tracking testing will be work

In [2]:
from boxmot import BotSort

# tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)

# Рабочий вариант как надо

In [ ]:
import cv2
import numpy as np

# Глобальные переменные
bbox = None
tracker = None
tracking = False

def mouse_callback(event, x, y, flags, param):
    global bbox, tracker, tracking
    if event == cv2.EVENT_LBUTTONDOWN:
        # При клике — задаём начальную область (например, 50x50 вокруг клика)
        size = 10
        bbox = (x - size//2, y - size//2, size, size)
        # Инициализируем трекер (используем CSRT — точный, но не самый быстрый)
        tracker = cv2.legacy.TrackerCSRT_create()
        tracking = True
        print('tracking ', tracking)

# Открываем камеру
cap = cv2.VideoCapture(0)

# Создаём окно и устанавливаем callback для мыши
cv2.namedWindow("Webcam")
cv2.setMouseCallback("Webcam", mouse_callback)
frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    

    if tracking and bbox is not None and tracker is not None:
        #if not tracker.isInitialized():
        succes_init = tracker.init(frame, bbox)
        if succes_init:
            print('Tracker initialized succesfuly!', frame_count)
        # else:
        #     print('Failed to initilize tracker!', frame_count)
        #     tracking = False
        #     tracker = None
        #     bbox = None
        #     break
        
        # Обновляем трекер
        
        succes, new_bbox = tracker.update(frame)
        
            
        if succes:
            print('succes ', succes)
            # Рисуем прямоугольник вокруг отслеживаемого объекта
            p1 = (int(new_bbox[0]), int(new_bbox[1]))
            p2 = (int(new_bbox[0] + new_bbox[2]), int(new_bbox[1] + new_bbox[3]))
            cv2.rectangle(frame, p1, p2, (0, 255, 0), 2)
            cv2.putText(frame, "Tracking", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        else:
            cv2.putText(frame, "Lost", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            tracking = False
        
    else:
        cv2.putText(frame, "Click to select target", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    frame_count +=1
    cv2.imshow("Webcam", frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [1]:
# global var
import cv2
import numpy as np

bbox = None
tracker = None
tracking = False

def mouse_callback(event, x, y, flags, param):
    global bbox, tracker, tracking

    if event == cv2.EVENT_LBUTTONDOWN:
        # задаем область вокруг клика
        size = 50
        bbox = (x - size//2, y - size//2, size, size)
        tracker = cv2.legacy.TrackerKCF_create()
        tracking = True


cap = cv2.VideoCapture(0)

cv2.namedWindow("self_tracking")
cv2.setMouseCallback('self_tracking', mouse_callback)

while True:
    ret, frame = cap.read()
    if not ret:
        print('Not read ret')
        break

    if tracking and bbox is not None:
        succes, new_box = tracker.update(frame)

        if new_box is not None:
            # рисуем прямоугольник вокруг объекта
            p1 = (int(new_box[0]), int(new_box[1]))
            p2 = (int(new_box[0] + new_box[2]), int(new_box[1] + new_box[3]))
            cv2.rectangle(frame, p1, p2, (0, 255, 0), 2)
            cv2.putText(frame, 'Traking self roi', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        else:
            cv2.putText(frame, 'Lost trackin', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            tracking = False

    else:
        cv2.putText(frame, "Click to select target", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow('self tracking', frame)

    if cv2.waitKey(1) &  0xFF == ord('q'):
        print('Press Q')
        break


cap.release()
cv2.destroyAllWindows()

Press Q


In [8]:
import cv2
import numpy as np
from boxmot import BotSort
from pathlib import Path
# tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)
# Глобальные переменные
bbox = None
tracker = None
tracking = False
selected_point = None

def mouse_callback(event, x, y, flags, param):
    global bbox, tracker, tracking, selected_point
    if event == cv2.EVENT_LBUTTONDOWN:
        selected_point = (x, y)
        # Создаем область вокруг точки для трекинга
        size = 50
        bbox = [x - size//2, y - size//2, size, size]
        
        tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)
        tracking = True
        print(f"Точка выбрана: {x}, {y}")

# Открываем камеру
cap = cv2.VideoCapture(0)

# Создаём окно и устанавливаем callback для мыши
cv2.namedWindow("Webcam")
cv2.setMouseCallback("Webcam", mouse_callback)

# Переменные для инициализации трекера
init_frame = None
init_bbox = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if tracking and bbox is not None and tracker is not None:
        try:
            # Преобразуем кадр в нужный формат
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Для инициализации трекера нам нужны детекции
            # Создаем искусственную детекцию вокруг выбранной точки
            x, y, w, h = bbox
            confidence = 0.9  # Высокая уверенность для ручного выбора
            
            # Создаем детекцию в формате [x1, y1, x2, y2, conf]
            detection = np.array([[x, y, x + w, y + h, confidence, 1]])
            
            # Обновляем трекер
            tracks = tracker.update(detection, rgb_frame)
            
            if len(tracks) > 0:
                # Получаем первый (и единственный) трек
                track = tracks[0]
                track_id = int(track[4])
                bbox_track = track[:4].astype(int)
                
                # Рисуем прямоугольник вокруг отслеживаемого объекта
                p1 = (bbox_track[0], bbox_track[1])
                p2 = (bbox_track[2], bbox_track[3])
                cv2.rectangle(frame, p1, p2, (0, 255, 0), 2)
                cv2.putText(frame, f"Tracking ID: {track_id}", (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                # Обновляем bbox для следующей итерации
                bbox = [bbox_track[0], bbox_track[1], 
                       bbox_track[2] - bbox_track[0], 
                       bbox_track[3] - bbox_track[1]]
                
            else:
                print(f'Not length track {len(tracks)}')
                cv2.putText(frame, "Lost", (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                tracking = False

        except Exception as e:
            print(f"Ошибка трекинга: {e}")
            tracking = False

    else:
        cv2.putText(frame, "Click to select target", (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Отображаем выбранную точку если есть
    if selected_point is not None:
        cv2.circle(frame, selected_point, 5, (0, 0, 255), -1)

    cv2.imshow("Webcam", frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

2025-09-25 16:23:46.208 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-25 16:23:46.209 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-25 16:23:46.516 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


Точка выбрана: 282, 339


In [1]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import BotSort

# --- Конфигурация ---
MODEL_PATH = r'weights_sdu_v3\best_sdu_v3.pt' #'yolo11n.pt'           # Легковесная модель YOLO
DEVICE = 'cpu'                      # Используем CPU, но можно 'cuda' если есть GPU
CLICK_RADIUS = 100                  # Радиус поиска объекта вокруг клика (в пикселях)
TRACKER_CONFIDENCE = 0.3            # Минимальная уверенность для трекинга

# --- Инициализация ---
model = YOLO(MODEL_PATH)            # Загружаем YOLOv8
tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)
   

# Глобальные переменные
selected_id = None                  # ID объекта, который мы выбираем кликом
last_click_pos = None               # Последняя позиция клика (x, y)

def mouse_callback(event, x, y, flags, param):
    global selected_id, last_click_pos
    if event == cv2.EVENT_LBUTTONDOWN:
        last_click_pos = (x, y)
        selected_id = None          # Сбрасываем старый ID, чтобы найти новый

# --- Открываем камеру ---
cap = cv2.VideoCapture(0)
cv2.namedWindow("Webcam - Click to Track")
cv2.setMouseCallback("Webcam - Click to Track", mouse_callback)

print("👉 Кликните мышью на объект, который хотите отслеживать.")
print("👉 Нажмите 'q' для выхода.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- 1. Детекция YOLO ---
    results = model(frame, verbose=False)  # verbose=False — не выводить логи
    dets = []
    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()      # [x1, y1, x2, y2]
        confs = r.boxes.conf.cpu().numpy()      # уверенность
        classes = r.boxes.cls.cpu().numpy()     # классы (0 = person, 1 = bike и т.д.)
        for box, conf, cls in zip(boxes, confs, classes):
            if conf >= TRACKER_CONFIDENCE:
                # Формат: [x1, y1, x2, y2, conf, cls]
                dets.append([box[0], box[1], box[2], box[3], conf, cls])

    dets = np.array(dets) if len(dets) > 0 else np.empty((0, 6))

    # --- 2. Трекинг через BoT-SORT ---
    if len(dets) > 0:
        tracks = tracker.update(dets, frame)
    else:
        tracks = np.empty((0, 7))  # [x1, y1, x2, y2, track_id, conf, cls]

    # --- 3. Если был клик — ищем ближайший объект ---
    if last_click_pos and len(tracks) > 0:
        click_x, click_y = last_click_pos
        min_dist = float('inf')
        closest_track = None

        for track in tracks:
            x1, y1, x2, y2, track_id = track[:5]
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            # Расстояние от клика до центра объекта
            dist = np.sqrt((center_x - click_x)**2 + (center_y - click_y)**2)

            # Если в радиусе и ближе — выбираем
            if dist < CLICK_RADIUS and dist < min_dist:
                min_dist = dist
                closest_track = track

        if closest_track is not None:
            selected_id = int(closest_track[4])  # track_id
            print(f"✅ Выбран объект с ID: {selected_id}")

        # Сбрасываем клик, чтобы не выбирать снова на каждом кадре
        last_click_pos = None

    # --- 4. Рисуем треки ---
    for track in tracks:
        x1, y1, x2, y2, track_id = track[:5]
        track_id = int(track_id)

        # Рисуем все треки серым
        color = (128, 128, 128)
        label = f"ID: {track_id}"

        if track_id == selected_id:
            # Выделенный объект — зелёный с рамкой
            color = (0, 255, 0)
            label += " (TRACKING)"

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, label, (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # --- 5. Подсказки ---
    if selected_id is None:
        cv2.putText(frame, "🖱️ Click to select object", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    else:
        cv2.putText(frame, f"🎯 Tracking ID: {selected_id}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # --- Показ ---
    cv2.imshow("Webcam - Click to Track", frame)

    # Выход по 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Очистка
cap.release()
cv2.destroyAllWindows()

2025-09-25 12:27:09.053 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-25 12:27:09.055 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-25 12:27:09.317 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


👉 Кликните мышью на объект, который хотите отслеживать.
👉 Нажмите 'q' для выхода.
✅ Выбран объект с ID: 6


var 3

In [3]:
import cv2
import numpy as np
from boxmot import BotSort  #BoTSORT

class BotSortTracker:
    def __init__(self):
        self.tracker = None
        self.tracking = False
        self.selected_point = None
        self.bbox = None
        self.track_id = None
        
    def mouse_callback(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            self.selected_point = (x, y)
            size = 60
            self.bbox = [x - size//2, y - size//2, size, size]
            
            # Инициализация BoT-SORT
            self.tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)
            # self.tracker = BoTSORT(
            #     model_weights=None,
            #     device='cpu',
            #     fp16=False,
            #     track_high_thresh=0.4,
            #     track_low_thresh=0.1,
            #     new_track_thresh=0.7,
            #     track_buffer=60,
            #     match_thresh=0.8,
            #     frame_rate=30
            # )
            self.tracking = True
            print(f"Инициализирован трекер для точки: {x}, {y}")
    
    def run(self):
        cap = cv2.VideoCapture(0)
        cv2.namedWindow("Webcam with BoT-SORT")
        cv2.setMouseCallback("Webcam with BoT-SORT", self.mouse_callback)
        
        frame_count = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            frame_count += 1
            
            if self.tracking and self.bbox is not None and self.tracker is not None:
                try:
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    
                    # Создаем детекцию для выбранной области
                    x, y, w, h = self.bbox
                    detection = np.array([[x, y, x + w, y + h, 0.95]])
                    
                    # Обновляем трекер каждые 2 кадра для производительности
                    # if frame_count % 2 == 0:
                    tracks = self.tracker.update(detection, rgb_frame)
                    # else:
                    #     # Используем предыдущие треки
                    #     tracks = self.tracker.trackers[0].tracks if hasattr(self.tracker.trackers[0], 'tracks') else []
                    
                    if len(tracks) > 0:
                        track = tracks[0]
                        self.track_id = int(track[4])
                        bbox_track = track[:4].astype(int)
                        
                        # Рисуем результаты
                        p1 = (bbox_track[0], bbox_track[1])
                        p2 = (bbox_track[2], bbox_track[3])
                        cv2.rectangle(frame, p1, p2, (0, 255, 0), 2)
                        
                        # Центр трека
                        center_x = (bbox_track[0] + bbox_track[2]) // 2
                        center_y = (bbox_track[1] + bbox_track[3]) // 2
                        cv2.circle(frame, (center_x, center_y), 3, (255, 0, 0), -1)
                        
                        cv2.putText(frame, f"BoT-SORT ID: {self.track_id}", 
                                  (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv2.putText(frame, f"Position: {center_x}, {center_y}", 
                                  (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                        
                        # Обновляем bbox
                        self.bbox = [bbox_track[0], bbox_track[1], 
                                   bbox_track[2] - bbox_track[0], 
                                   bbox_track[3] - bbox_track[1]]
                    else:
                        cv2.putText(frame, "Tracking Lost", (10, 30), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        self.tracking = False
                        
                except Exception as e:
                    print(f"Ошибка: {e}")
                    self.tracking = False
            
            # Показываем инструкцию и выбранную точку
            if not self.tracking:
                cv2.putText(frame, "Click to select tracking point", 
                          (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            if self.selected_point is not None:
                cv2.circle(frame, self.selected_point, 6, (0, 0, 255), -1)
                cv2.putText(frame, "Selected", 
                          (self.selected_point[0] + 10, self.selected_point[1]), 
                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
            
            cv2.imshow("Webcam with BoT-SORT", frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()

# Запуск
if __name__ == "__main__":
    tracker = BotSortTracker()
    tracker.run()

2025-09-25 12:35:09.116 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-25 12:35:09.116 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-25 12:35:09.427 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


Инициализирован трекер для точки: 344, 253
Ошибка: Unsupported 'dets' 2nd dimension lenght, valid lenghts is 6 (x1,y1,x2,y2,conf,cls)
